In [2]:
import numpy as np
from keras import layers
from keras.layers import Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization, Flatten, Conv2D, AveragePooling2D, MaxPooling2D, GlobalMaxPool2D
from keras.models import Model, load_model
from keras.preprocessing import image
from tensorflow.python.keras.utils import layer_utils
from keras.applications.imagenet_utils import preprocess_input
import pydot
from IPython.display import SVG
from keras.utils import model_to_dot
from keras.utils import plot_model
from resnets_utils import *
from keras.initializers import glorot_uniform
import scipy.misc
from matplotlib.pyplot import imshow
%matplotlib inline

import keras.backend  as K
K.set_image_data_format('channels_last')
# k.set_learning_phase(1)

# The Identity Block

In [3]:
def identity_block(X, f, filters, stage, block):
    """
    Implementation of the identity block as defined

    Arguments:
    X -- input tensor of shape (m, n_H_prev, n_W_prev, n_C_prev)
    f -- integer, specifying the shape of the middle CONV's window for the main path
    filters -- python list of integers, defining the number of filters in the CONV layers of the main path
    stage -- integer, used to name the layers, depending on their position in the network
    block -- string/character,used to name the layers, depending on their position in the network

    Returns:
    X -- output of the identity block, tensor of shape (n_H, n_W, n_C)
    """

    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    F1, F2, F3 = filters

    X_shortcut = X

    X = Conv2D(filters=F1, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base+'2a', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation("relu")(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding="same", name=conv_name_base+'2b', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation("relu")(X)

    X = Conv2D(filters=F3, kernel_size=(1, 1), strides=(1, 1), padding="valid", name=conv_name_base + '2c', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)

    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X

In [5]:
np.random.seed(1)
X = np.random.randn(3, 4, 4, 6).astype("float32")

A = identity_block(X, f=2, filters=[2, 4, 6], stage=1, block='a')

print("out = " + str(A.numpy()[1, 1, 0]))

out = [0.        0.        1.3454674 2.031818  0.        1.3246754]


# The Convolution Block

In [6]:
def convolutional_block(X, f, filters, stage, block, s = 2):
    """
    Implementation of the convolutional block

    Arguments:
    X -- input tensor of shape (m, n_H_prev, n_W_prev, n_C_prev)
    f -- integer, specifying the shape of the middle CONV's window for the main path
    filters -- python list of integers, defining the number of filters in the CONV layers of the main part
    stage -- integer, used to name the layers, depending on their position in the network
    block -- string/character, used to name the layers, depending on their position in the network
    s -- integer, specifying the stride to be used

    Returns:
    X -- output of the convolutional block, tensor of shape (n_H, n_W, n_C)
    """

    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    F1, F2, F3 = filters

    X_shortcut = X

    X = Conv2D(filters=F1, kernel_size=(1,1), strides=(s, s), padding="valid", name=conv_name_base + '2a', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding="same", name=conv_name_base + '2b', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F3, kernel_size=(1,1), strides=(1,1), padding="valid", name=conv_name_base + '2c', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)

    X_shortcut = Conv2D(filters=F3, kernel_size=(1,1), strides=(s,s), padding="valid", name=conv_name_base + '1', kernel_initializer=glorot_uniform(seed=0))(X_shortcut)
    X_shortcut = BatchNormalization(axis=3, name=bn_name_base + '1')(X_shortcut)

    X = Add()([X, X_shortcut])
    X = Activation("relu")(X)

    return X

In [7]:
np.random.seed(1)
X = np.random.randn(3, 4, 4, 6).astype("float32")

A = convolutional_block(X, f=2, filters=[2, 4, 6], stage=1, block='a')

print("out = " + str(A.numpy()[1, 1, 0]))

out = [0.         0.         0.         0.8929657  0.         0.19882731]


In [8]:
def ResNet50(input_shape = (64, 64, 3), classes = 6):
    """
    Implementation of the popular ResNet50 the following architecture:
    CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL -> TOPLAYER

    Arguments:
    input_shape -- shape of the images of the dataset
    classes -- integer, number of classes

    Returns:
    model -- a model() instance in keras
    
    """

    X_input = Input(input_shape)

    X = ZeroPadding2D(padding=(3,3))(X_input)

    # Stage 1
    X = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), name='conv1', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name="bn_conv1")(X)
    X = MaxPooling2D(pool_size=(3,3), strides=(2, 2))(X)

    # Stage 2
    X = convolutional_block(X, f=3, filters=[64, 64, 256], stage=2, s=1, block='a')
    X = identity_block(X, f=3, filters=[64, 64, 256], stage=2, block='b')
    X = identity_block(X, f=3, filters=[64, 64, 256], stage=2, block='c')

    # Stage 3
    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, s=2, block='a')
    X = identity_block(X, f=3, filters=[128, 128, 512], stage=3, block='b')
    X = identity_block(X, f=3, filters=[128, 128, 512], stage=3, block='c')
    X = identity_block(X, f=3, filters=[128, 128, 512], stage=3, block='d')

    # Stage 4
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, s=2, block='a')
    X = identity_block(X, f=3, filters=[256, 256, 1024], stage=4, block='b')
    X = identity_block(X, f=3, filters=[256, 256, 1024], stage=4, block='c')
    X = identity_block(X, f=3, filters=[256, 256, 1024], stage=4, block='d')
    X = identity_block(X, f=3, filters=[256, 256, 1024], stage=4, block='e')
    X = identity_block(X, f=3, filters=[256, 256, 1024], stage=4, block='f')

    # Stage 5
    X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, s=2, block='a')
    X = identity_block(X, f=3, filters=[512, 512, 2048], stage=5, block='b')
    X = identity_block(X, f=3, filters=[512, 512, 2048], stage=5, block='c')

    X = AveragePooling2D(pool_size=(2, 2), name="avg_pool")(X)
    X = Flatten()(X)
    X = Dense(classes, activation="softmax", name='fc' + str(classes), kernel_initializer=glorot_uniform(seed=0))(X)

    model = Model(inputs=X_input, outputs = X, name='ResNet50')

    return model

In [19]:
model = ResNet50(input_shape=(64, 64, 3), classes=6)

In [20]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [11]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_dataset()

X_train = X_train_orig / 255.
X_test = X_test_orig / 255.

Y_train = convert_to_one_hot(Y_train_orig, 6).T
Y_test = convert_to_one_hot(Y_test_orig, 6).T

print("Number of training examples = " + str(X_train.shape[0]))
print("Number of test examples = " + str(X_test.shape[0]))
print("X_train shape: " + str(X_train.shape))
print("Y_train shape: " + str(Y_train.shape))
print("X_test shape: " + str(X_test.shape))
print("Y_test shape: " + str(Y_test.shape))

Number of training examples = 1080
Number of test examples = 120
X_train shape: (1080, 64, 64, 3)
Y_train shape: (1080, 6)
X_test shape: (120, 64, 64, 3)
Y_test shape: (120, 6)


In [21]:
model.fit(X_train, Y_train, epochs=10, batch_size=32)

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 130s 2s/step - accuracy: 0.4731 - loss: 1.9059
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.8019 - loss: 0.6008
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.8759 - loss: 0.4158
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 61s 2s/step - accuracy: 0.9194 - loss: 0.2455
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.9083 - loss: 0.2730
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.9648 - loss: 0.1096
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.9648 - loss: 0.1037
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.9444 - loss: 0.1726
Epoch 9/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 61s 2s/step - accuracy: 0.9602 - loss: 0.1191
Epoch 10/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 61s 2s/step - accuracy: 0.9778 - loss: 0.0819


In [22]:
predictions = model.evaluate(X_test, Y_test)
print("loss = " + str(predictions[0]))
print("Test accuracy = " + str(predictions[1]))

4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 270ms/step - accuracy: 0.4417 - loss: 2.7651
loss = 2.765141487121582
Test accuracy = 0.4416666626930237
